In [2]:

!pip list


Package                 Version
----------------------- -----------
absl-py                 2.5.0
asttokens               3.0.2
astunparse              1.6.3
blinker                 1.9.0
certifi                 2026.7.22
charset-normalizer      3.5.2
click                   8.5.0
cloudpickle             3.1.2
colorama                0.4.6
comm                    0.2.3
debugpy                 1.8.22
executing               2.2.1
Flask                   3.1.3
flatbuffers             25.12.19
gast                    0.7.0
google-pasta            0.2.0
grpcio                  1.84.0
h5py                    3.14.0
idna                    3.20
ipykernel               7.4.0
ipython                 9.17.1
ipython_pygments_lexers 1.1.1
itsdangerous            2.2.0
jedi                    0.20.0
Jinja2                  3.1.6
joblib                  1.6.0
jupyter_client          8.10.0
jupyter_core            5.9.1
keras                   3.15.1
libclang                18.1.1
markdown-it-py    

In [1]:
import pandas as pd
import matplotlib as pl
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
import joblib

In [5]:
df = pd.read_csv("data/historical_quality_data.csv")
df.head(5)

,product_id,temperature,pressure,vibration,speed,weight,actual_result
0,HIST-00001,71.25,39.31,2.46,125.97,499.23,PASS
1,HIST-00002,76.65,42.18,3.46,121.11,498.61,DEFECT
2,HIST-00003,68.16,35.94,2.91,131.14,500.25,PASS
3,HIST-00004,70.48,36.54,4.36,137.69,498.63,DEFECT
4,HIST-00005,76.07,35.55,3.31,107.13,484.28,DEFECT


In [6]:
df.shape

(3000, 7)

In [7]:
df.info

<bound method DataFrame.info of       product_id  temperature  pressure  vibration   speed  weight  \
0     HIST-00001        71.25     39.31       2.46  125.97  499.23   
1     HIST-00002        76.65     42.18       3.46  121.11  498.61   
2     HIST-00003        68.16     35.94       2.91  131.14  500.25   
3     HIST-00004        70.48     36.54       4.36  137.69  498.63   
4     HIST-00005        76.07     35.55       3.31  107.13  484.28   
...          ...          ...       ...        ...     ...     ...   
2995  HIST-02996        74.70     47.01       1.88  120.13  495.05   
2996  HIST-02997        66.56     37.29       1.14  107.33  497.36   
2997  HIST-02998        74.61     39.20       3.54  115.87  508.52   
2998  HIST-02999        73.94     43.31       2.78  130.98  493.70   
2999  HIST-03000        74.26     34.70       2.41  138.43  489.18   

     actual_result  
0             PASS  
1           DEFECT  
2             PASS  
3           DEFECT  
4           DEFECT  
.

In [8]:
df.describe()

,temperature,pressure,vibration,speed,weight
count,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000
mean,72.027360,40.025480,2.608587,119.951570,499.953693
std,5.171944,4.007571,1.211138,8.460901,6.042825
min,54.890000,26.680000,0.200000,89.950000,478.900000
25%,68.597500,37.270000,1.750000,114.212500,495.887500
50%,72.080000,40.090000,2.610000,120.015000,499.915000
75%,75.440000,42.840000,3.430000,125.702500,504.080000
max,89.500000,51.690000,6.760000,151.290000,518.990000


In [9]:
df["actual_result"].value_counts(normalize=True)*100

actual_result
PASS      61.5
DEFECT    38.5
Name: proportion, dtype: float64

In [10]:
X=df.drop(columns=["actual_result","product_id"])
y=df["actual_result"].map({"PASS":0,"DEFECT":1})

In [11]:
X_test, X_train, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [12]:
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(X_train)
x_test_scaled = scaler.transform(X_test)

In [13]:
model = keras.models.Sequential([
    # keras.layers.Input(shape=(5,))
    keras.layers.Input(shape=(X.shape[1],)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 241 (964.00 B)

 Trainable params: 241 (964.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
import warnings
warnings.filterwarnings("ignore")

In [15]:
model.compile(
    loss="binary_crossentropy",
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    metrics=["accuracy"]

)

In [16]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [17]:
history = model.fit(
    x_train_scaled,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping]



)

'''
3000 total
2400  train area
  1920 training real
   480-->validation
'''

Epoch 1/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.5979 - loss: 0.6785 - val_accuracy: 0.5667 - val_loss: 0.6896
Epoch 2/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6021 - loss: 0.6747 - val_accuracy: 0.5750 - val_loss: 0.6907
Epoch 3/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5979 - loss: 0.6718 - val_accuracy: 0.5750 - val_loss: 0.6901
Epoch 4/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6146 - loss: 0.6694 - val_accuracy: 0.5667 - val_loss: 0.6907
Epoch 5/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6125 - loss: 0.6672 - val_accuracy: 0.5750 - val_loss: 0.6894
Epoch 6/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6208 - loss: 0.6653 - val_accuracy: 0.5917 - val_loss: 0.6882
Epoch 7/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6250 - loss: 0.6638 - val_accuracy: 0.5917 - val_loss: 0.6880
Epoch 8/50
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6250 - loss: 0.6624 - val_accuracy: 0.5917 - va

'\n3000 total\n2400  train area\n  1920 training real\n   480-->validation\n'

In [18]:
model.save("model.keras")

In [19]:
joblib.dump(scaler, "scaler.joblib")

['scaler.joblib']